In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torch.utils.data import TensorDataset, DataLoader
from typing import Optional, Callable, Dict
from tqdm.notebook import tqdm, trange
from torchvision import datasets, transforms, torchvision
import numpy as np
import plotly.express as px
import random

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(0)
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("device:", device)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("device:", device)

To generate my data (points on a circle) I sample the angle $\theta$ from a uniform in $[0,2\pi]$ and use polar coordinates from a radius $r$:
- $x = r\cdot cos(\theta)$
- $y = r\cdot sin(\theta)$

In [ ]:
# initial dataset and data loader
def circle(num_samples=5000, radius=2.0):
    theta = torch.rand(num_samples) * 2 * math.pi
    x = radius * torch.cos(theta)
    y = radius * torch.sin(theta)
    return torch.stack((x, y), dim=-1)

x_clean_train = circle(num_samples=10000)
train_ds = TensorDataset(x_clean_train)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True) 

### ARCHITECTURE
Input: a point $(x,y)$

Output: the predicted noise $(\epsilon_x, \epsilon_y)$

From the predicted noise we're defining a direction because when we substract it we move closer to the clean data.

In [ ]:

class ScoreMLP2D(nn.Module):
    def __init__(self, 
                 num_layers: int, 
                 hidden_dim: int,
                 activation: Callable[[torch.Tensor], torch.Tensor]) -> None:
        super().__init__()
        self.first_layer = nn.Linear(in_features=2, out_features=hidden_dim)
        self.layers = nn.ModuleList() 
        for i in range(num_layers):
            self.layers.append(
                nn.Linear(in_features=hidden_dim, out_features=hidden_dim)
            )
        self.activation = activation
        self.last_layer = nn.Linear(in_features=hidden_dim, out_features=2)

    def forward(self, meshgrid: torch.Tensor) -> torch.Tensor:
        out = meshgrid
        out = self.first_layer(out)
        for layer in self.layers:
            out = layer(out)
            out = self.activation(out)
        out = self.last_layer(out)
        return out



In [ ]:
#initial state and training parameters
model = ScoreMLP2D(num_layers=3, hidden_dim=128, activation=torch.nn.functional.elu)
model = model.to(device)
opt = optim.Adam(model.parameters(), lr=0.001)
sigma = 0.3 
epochs = 6#50

In [ ]:
for epoch in tqdm(range(epochs), desc="epoch"):
    model.train()
    epoch_loss = 0.0  
    num_batches = 0   
    for (xb,) in train_dl:
        x_clean = xb.to(device)
        noise = torch.randn_like(x_clean)
        # here I dirty the data with noise
        x_noisy = x_clean + sigma * noise
        pred_noise = model(x_noisy)
        #I compute the mse loss
        loss = F.mse_loss(pred_noise, noise)
        loss.backward()
        opt.step()
        opt.zero_grad()
        epoch_loss += loss.item()
        num_batches += 1


    # mean loss in epoch
    print(epoch, epoch_loss / num_batches)
    #if (epoch + 1) % 10 == 0 or epoch == 0:
    #    print(epoch, epoch_loss / num_batches)

Given the equations in the pdf I can get the score as a function of the noise.

- $\nabla_{\tilde{\mathbf{x}}} \log q_\sigma(\tilde{\mathbf{x}} \mid \mathbf{x}) = \frac{1}{\sigma^2}(\mathbf{x} - \tilde{\mathbf{x}})$

- $\tilde{\mathbf{x}} = \mathbf{x} + \sigma \epsilon \implies \mathbf{x} - \tilde{\mathbf{x}} = -\sigma \epsilon$

Here:
* **$\mathbf{x}$** is the original data
* **$\tilde{\mathbf{x}}$** represents noisy data
* **$\sigma$** is the SD of the Gaussian noise
* **$\epsilon$** is the standard Gaussian noise vector that will dirty my data
* **$q_\sigma(\tilde{\mathbf{x}} \mid \mathbf{x})$** is the conditional probability distribution of the noisy data given the clean data and will model the noise corruption process.
* **$\nabla_{\tilde{\mathbf{x}}}$**: the gradient is computed wrt the noisy data and provides the direction of change

In the end I get
$$\nabla_{\tilde{\mathbf{x}}} \log q_\sigma(\tilde{\mathbf{x}} \mid \mathbf{x}) = \frac{-\sigma \epsilon}{\sigma^2} = -\frac{\epsilon}{\sigma}$$

Then I implement Langevin dynamics `x_langevin` from the following equation
$$\tilde{\mathbf{x}}^k = \tilde{\mathbf{x}}^{k-1} + \frac{\lambda_i}{2} \mathbf{s}_\theta(\tilde{\mathbf{x}}^{k-1}, \sigma_i) + \sqrt{\lambda_i}\mathbf{z}^k$$

where at each step $k$ I update the previous position $\tilde{\mathbf{x}}^{k-1}$ moving along the direction of the score with a step length scaled by $\lambda_i$. A standard Gaussian vector $\mathbf{z}^k$ adds randomness to explore the space.

In [ ]:
# langevin sampling with snapshots
model.eval()
num_steps = 300
step_size = 0.01  

x_langevin = torch.randn(1000, 2, device=device) * 4.0 
initial_noise_np = x_langevin.cpu().clone().numpy()

snapshots = {}
monitored_steps = [1, 10, 20, 50, 100, 300]

with torch.no_grad():
    for k in range(num_steps):
        pred_noise = model(x_langevin)
        score = - pred_noise / sigma


        # langevin dynamics step
        z = torch.randn_like(x_langevin)
        x_langevin = x_langevin + 0.5 * step_size * score + math.sqrt(step_size) * z


        current_step = k + 1
        if current_step in monitored_steps:
            snapshots[current_step] = x_langevin.cpu().numpy()

final_generated_np = x_langevin.cpu().numpy()

In [ ]:
#visualization
x_clean_np = x_clean_train.cpu().numpy()
def plot_2d_points(points, title, color):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=points[:, 0], y=points[:, 1],
            mode='markers',
            marker=dict(size=4, color=color, opacity=0.6),
            name=title
        )
    )
    fig.update_layout(
        title=title, width=600, height=600,
        xaxis=dict(range=[-6, 6]), yaxis=dict(range=[-6, 6])
    )
    fig.show()

plot_2d_points(x_clean_np, "Original data", "lightgreen")
plot_2d_points(initial_noise_np, "Noise", "red")
plot_2d_points(final_generated_np, "Data generated", "orange")

In [ ]:
#data generation process
plot_2d_points(initial_noise_np, "Step 0: Pure Noise", "red")
for step in monitored_steps:
    plot_2d_points(
        snapshots[step], 
        title=f"Step {step}: Inversion Process", 
        color="orange"
    )

In [ ]:
#better representation in a grid
all_steps = monitored_steps
titles = [f"Step {step}" for step in all_steps]

fig = make_subplots(rows=2, cols=3, subplot_titles=titles)

for idx, step in enumerate(all_steps):
    row = (idx // 3) + 1
    col = (idx % 3) + 1
    points = initial_noise_np if step == 0 else snapshots[step]


    fig.add_trace(
        go.Scatter(
            x=points[:, 0], y=points[:, 1],
            mode='markers',
            marker=dict(size=3, color="orange", opacity=0.6),
            showlegend=False
        ),
        row=row, col=col
    )

fig.update_xaxes(range=[-6, 6])
fig.update_yaxes(range=[-6, 6])

fig.show()

Let's see if it generalizes: let's generate sinusoids

In [ ]:
def sines(num_samples=5000, amp=1.0, freq=1.0):
    theta = torch.rand(num_samples) * 2 * math.pi
    y = amp * torch.sin(freq * theta)
    
    return torch.stack((theta, y), dim=-1)

x_clean_sine = sines(num_samples=10000)
print(x_clean_sine.shape)  
train_ds_sine = TensorDataset(x_clean_sine)
train_dl_sine = DataLoader(train_ds_sine, batch_size=64, shuffle=True) 

In [ ]:
#initial state and training parameters
model = ScoreMLP2D(num_layers=3, hidden_dim=128, activation=torch.nn.functional.elu)
model = model.to(device)
opt = optim.Adam(model.parameters(), lr=0.001)
sigma = 0.3 
epochs = 21
losses_h = []
for epoch in tqdm(range(epochs), desc="epoch"):
    model.train()
    epoch_loss = 0.0  
    num_batches = 0   
    for (xb,) in train_dl_sine:
        x_clean = xb.to(device)
        noise = torch.randn_like(x_clean)
        # here I dirty the data with noise
        x_noisy = x_clean + sigma * noise
        pred_noise = model(x_noisy)
        #I compute the mse loss
        loss = F.mse_loss(pred_noise, noise)
        loss.backward()
        opt.step()
        opt.zero_grad()
        epoch_loss += loss.item()
        num_batches += 1


    # mean loss in epoch
    mean_epoch_loss = epoch_loss / num_batches
    losses_h.append(mean_epoch_loss)
    print(epoch, mean_epoch_loss)
    #if (epoch + 1) % 10 == 0 or epoch == 0:
    #    print(epoch, epoch_loss / num_batches)
    
# just some monitoring
min_loss_index = losses_h.index(min(losses_h))
print(f"Minimum loss of {losses_h[min_loss_index]:.6f} at epoch {min_loss_index}")
#loss plot
loss_fig = go.Figure()
loss_fig.add_trace(
    go.Scatter(
        x=list(range(epochs)), 
        y=losses_h, 
        mode='lines+markers', 
        name='MSE Loss',
        line=dict(color='deepskyblue', width=2)
    )
)
loss_fig.update_layout(
    title="Training Loss",
    xaxis_title="Epoch",
    yaxis_title="MSE Loss"
)
loss_fig.show()

# langevin sampling with snapshots
model.eval()
num_steps = 1000
step_size = 0.01  

x_langevin = torch.randn(1000, 2, device=device) * 4.0 
initial_noise_np = x_langevin.cpu().clone().numpy()

snapshots = {}
monitored_steps = [1, 10, 20, 50, 100, 300, 500, 1000]

with torch.no_grad():
    for k in range(num_steps):
        pred_noise = model(x_langevin)
        score = - pred_noise / sigma


        # langevin dynamics step
        z = torch.randn_like(x_langevin)
        x_langevin = x_langevin + 0.5 * step_size * score + math.sqrt(step_size) * z


        current_step = k + 1
        if current_step in monitored_steps:
            snapshots[current_step] = x_langevin.cpu().numpy()

final_generated_np = x_langevin.cpu().numpy()
all_steps =  monitored_steps
titles = [f"Step {step}" for step in all_steps]

fig = make_subplots(rows=2, cols=4, subplot_titles=titles)

for idx, step in enumerate(all_steps):
    row = (idx // 4) + 1
    col = (idx % 4) + 1
    points = initial_noise_np if step == 0 else snapshots[step]


    fig.add_trace(
        go.Scatter(
            x=points[:, 0], y=points[:, 1],
            mode='markers',
            marker=dict(size=3, color="orange", opacity=0.6),
            showlegend=False
        ),
        row=row, col=col
    )

fig.update_xaxes(range=[-6, 6])
fig.update_yaxes(range=[-6, 6])

fig.show()

## With different values of $\sigma$

In [ ]:
## see torch.nn.functional.elu vs F.elu
sigmas = [0.1, 0.3, 0.8]
epochs = 21
my_models = {}
my_losses = {}
def energy(y_pred, y_true):
    return torch.nn.functional.mse_loss(y_pred, y_true)
for sigma in sigmas:
    print(f"RUNNING EXPERIMENT WITH SIGMA = {sigma} ")
    # init
    model = ScoreMLP2D(num_layers=3, hidden_dim=128, activation=torch.nn.functional.elu )
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=0.001)
    losses_h = []
    # training
    for epoch in tqdm(range(epochs), desc=f"Training (sigma={sigma})"):
        model.train()
        epoch_loss = 0.0
        samples_nr = 0
        for (xb,) in train_dl_sine:
            x_clean = xb.to(device)
            noise = torch.randn_like(x_clean)
            x_noisy = x_clean + sigma * noise
            pred_noise = model(x_noisy)
            loss = energy(pred_noise, noise)
            loss.backward()
            opt.step()
            opt.zero_grad()
            epoch_loss += loss.item() * len(x_clean)
            samples_nr += len(x_clean)
        mean_epoch_loss = epoch_loss / samples_nr
        losses_h.append(mean_epoch_loss)
    my_models[sigma] = model.state_dict()
    my_losses[sigma] = losses_h
    min_loss_idx = losses_h.index(min(losses_h))
    print(f"[Sigma {sigma}] min loss {losses_h[min_loss_idx]:.6f} at epoch {min_loss_idx}")
    # training loss plot
    loss_fig = go.Figure()
    loss_fig.add_trace(
        go.Scatter(
            x=list(range(epochs)),
            y=losses_h,
            mode='lines+markers',
            name=f'MSE Loss (sigma={sigma})',
            line=dict(color='deepskyblue', width=2)
        )
    )
    loss_fig.update_layout(
        title=f"Training Loss Curve (Sigma = {sigma})",
        xaxis_title="Epoch", yaxis_title="MSE Loss"
    )
    loss_fig.show()
    # langevin sampling
    model.eval()
    num_steps = 1000
    step_size = 0.01
    x_langevin = torch.randn(1000, 2, device=device) * 4.0
    snapshots = {}
    monitored_steps = [1, 10, 20, 50, 100, 300, 500, 1000]
    with torch.no_grad():
        for k in range(num_steps):
            pred_noise = model(x_langevin)
            score = - pred_noise / sigma
            z = torch.randn_like(x_langevin)
            x_langevin = x_langevin + 0.5 * step_size * score + math.sqrt(step_size) * z
            current_step = k + 1
            if current_step in monitored_steps:
                snapshots[current_step] = x_langevin.cpu().numpy()
    all_steps = monitored_steps
    titles = [f"Step {step}" for step in all_steps]
    grid_fig = make_subplots(rows=2, cols=4, subplot_titles=titles)
    for idx, step in enumerate(all_steps):
        row = (idx // 4) + 1
        col = (idx % 4) + 1
        points = snapshots[step]
        grid_fig.add_trace(
            go.Scatter(
                x=points[:, 0], y=points[:, 1],
                mode='markers',
                marker=dict(size=3, color="orange", opacity=0.6),
                showlegend=False
            ),
            row=row, col=col
        )
    grid_fig.update_layout(
        title_text=f"Langevin Inversion Grid (Sigma = {sigma})"
    )
    grid_fig.update_xaxes(range=[-6, 6])
    grid_fig.update_yaxes(range=[-6, 6])
    grid_fig.show()

## Let's go NCSNs!
Since basic score estimation is unreliable in low density regions now we're conditioning on $\sigma$ instead of having it fixed. Then I use Langevin dynamics running a schedule to decrease sigma progressively.

In [ ]:
class CondMLP(nn.Module):
    def __init__(self,
                num_layers: int,
                hidden_dim: int,
                activation: Callable[[torch.Tensor], torch.Tensor]) -> None:
        super().__init__()
        self.first_layer = nn.Linear(in_features=2, out_features=hidden_dim)
        self.sigma_layer1 = nn.Linear(in_features=1, out_features=hidden_dim)
        self.sigma_layer2 = nn.Linear(in_features=hidden_dim, out_features=hidden_dim)
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(
                nn.Linear(in_features=hidden_dim, out_features=hidden_dim)
            )
        self.activation = activation
        self.last_layer = nn.Linear(in_features=hidden_dim, out_features=2)
    def forward(self, meshgrid: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
        out = meshgrid
        out = self.first_layer(out)
        #inject noise
        h_sigma = self.sigma_layer2(self.activation(self.sigma_layer1(sigma)))
        out += h_sigma
        out = self.activation(out)
        for layer in self.layers:
            out = layer(out)
            out = self.activation(out)
        out = self.last_layer(out)
        return out

In [ ]:
#noise schedule
L = 8  
sigma_max = 4.0
sigma_min = 0.05
sigmas = torch.exp(torch.linspace(math.log(sigma_max), math.log(sigma_min), L)).to(device)
model = CondMLP(num_layers=3, hidden_dim=128, activation=torch.nn.functional.elu).to(device)
opt = optim.Adam(model.parameters(), lr=0.001)
epochs = 30
losses_h = []
#training
for epoch in tqdm(range(epochs), desc="Training NCSN"):
    model.train()
    epoch_loss = 0.0  
    samples_nr = 0   
    for (xb,) in train_dl_sine:
        x_clean = xb.to(device)
        batch_size = x_clean.shape[0]
        indices = torch.randint(0, L, (batch_size,), device=device)
        batch_sigmas = sigmas[indices].view(-1, 1) 
        noise = torch.randn_like(x_clean)
        x_noisy = x_clean + batch_sigmas * noise
        pred_noise = model(x_noisy, batch_sigmas)
        loss = F.mse_loss(pred_noise, noise)
        loss.backward()
        opt.step()
        opt.zero_grad()
        epoch_loss += loss.item() * len(x_clean)
        samples_nr += len(x_clean)
    losses_h.append(epoch_loss / samples_nr)
print(f"Final loss: {losses_h[-1]:.6f}")

In [ ]:
# annealed langevin sampling
model.eval()
steps_per_scale = 125 
epsilon = 0.005       
x_langevin = torch.randn(1000, 2, device=device) * sigmas[0]
snapshots = {}
snapshot_titles = []
with torch.no_grad():
    for i, sig in enumerate(sigmas):
        step_size = epsilon * (sig / sigmas[-1])**2
        for k in range(steps_per_scale):
            batch_sig = torch.full((1000, 1), sig.item(), device=device)
            pred_noise = model(x_langevin, batch_sig)
            score = -pred_noise / sig
            z = torch.randn_like(x_langevin)
            x_langevin = x_langevin + 0.5 * step_size * score + math.sqrt(step_size) * z
        title = f"Scale {i+1} (sigma={sig:.2f})"
        snapshots[title] = x_langevin.cpu().numpy()
        snapshot_titles.append(title)

In [ ]:
#viz
fig = make_subplots(rows=2, cols=4, subplot_titles=snapshot_titles)
for idx, (title, points) in enumerate(snapshots.items()):
    row = (idx // 4) + 1
    col = (idx % 4) + 1
    fig.add_trace(
        go.Scatter(
            x=points[:, 0], y=points[:, 1],
            mode='markers',
            marker=dict(size=3, color="orange", opacity=0.6),
            showlegend=False
        ),
        row=row, col=col
    )
fig.update_layout(title_text="Annealed Langevin Dynamics (NCSN)")
fig.update_xaxes(range=[-6, 6])
fig.update_yaxes(range=[-6, 6])
fig.show()

## Diffusion Probabilistic Models

DPMs are based on a Markov Chain made by a sequence of states. Starting from a clean sample taken from the true distribution the forward process adds Gaussian noise for $T=1000$ steps. The main properties of MC is that the future is independent from the past given the present and in this case the final state will be a standard Gaussian distribution (pure noise).
Data generation is allowed by reverting this process: starting from random noise an update rule iteratively denoise it untill we get to new data. In each transition some random noise is added to explore the space of the data.
Since the network has no information on the temporal progression and a simple discrete integer timestep won't add much expressivity, that same timestep is transformed into sines and cosines, continuous functions with good properties as we will see.

## SinusoidalPositionEmbeddings


As we've seen in transformers architectures tokens are processed in parallel, so order is not tracked. Sentences as "Luca slaps Giovanni" and "Giovanni slaps Luca" are considered the same. In the paper "Attention is all you need" to track token order positional encodings (PEs) are injected into the encoder and decoder adding them to the input embeddings. PEs are implemented from sine and cosine functions with different frequencies. They have good properties: since they're boundend in $[-1, 1]$ they avoid numerical explosion; being periodical they can manage variable length sequences; the token relative distance is constant either if I consider tokens 1-2 or 2000-2001

- $PE_{(pos,\, 2i)} = \sin\left(\frac{pos}{10000^{\frac{2i}{d_m}}}\right)$

- $PE_{(pos,\, 2i+1)} = \cos\left(\frac{pos}{10000^{\frac{2i}{d_m}}}\right)$

where $pos$ is the token position in the sequence and $d_m$ is the embedding space dimension.

*Sources*
- [Attention is all you need](https://arxiv.org/pdf/1706.03762)
- [Sinusoidal positional encodings](https://medium.com/@pranay.janupalli/understanding-sinusoidal-positional-encoding-in-transformers-26c4c161b7cc)

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    #pos = time equivalently
    def forward(self, pos):
        half_dim = self.dim // 2
        inv_freq = 1.0 / (10000 ** (torch.arange(0, half_dim, device=pos.device).float() / half_dim))
        args = pos[:, None] * inv_freq[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)